In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na requirements.
Inirerekomenda namin ang paggamit ng mga bersyong ito o mas bago pa.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Bukod sa [pag-visualize ng mga instruksyon sa isang circuit](/guides/visualize-circuits), maaari ka ring mag-visualize ng scheduling ng isang circuit sa pamamagitan ng Qiskit [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) method. Makakatulong ang visualization na ito para mabilis mong makita ang idling time sa mga qubit, halimbawa. Gayunpaman, hindi nagbabalik ng tumpak na resulta ang method na ito para sa mga dynamic na circuit. Para ma-visualize ang dynamic circuit scheduling, gamitin ang `draw_circuit_schedule_timing` method, gaya ng inilarawan sa seksyong [Suporta ng Qiskit Runtime](#qr-support).

## Mga Halimbawa

Para ma-visualize ang isang naka-schedule na circuit program, maaari mong tawagin ang function na ito gamit ang isang set ng mga control argument. Karamihan sa hitsura ng output image ay maaaring baguhin ng isang stylesheet, ngunit hindi ito kinakailangan.

### Gumuhit gamit ang default na stylesheet

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![Output ng nakaraang code cell](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### Gumuhit gamit ang stylesheet para sa pag-debug ng program

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![Output ng nakaraang code cell](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

Maaari kang lumikha ng mga custom na generator o layout function at i-update ang isang umiiral na stylesheet gamit ang mga custom na function. Sa ganitong paraan, kontrolado mo ang karamihan sa hitsura ng output image nang hindi binabago ang codebase ng scheduled circuit drawer. Tingnan ang [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) API reference para sa higit pang mga halimbawa.
<span id="qr-support"></span>
## Suporta ng Qiskit Runtime
Habang ang timeline drawer na built-in sa Qiskit ay kapaki-pakinabang para sa mga static na circuit, maaaring hindi ito tumpak na maipakita ang timing ng mga [dynamic na circuit](/guides/classical-feedforward-and-control-flow) dahil sa mga implicit na operasyon tulad ng broadcasting at branch determination. Bilang bahagi ng suporta sa dynamic na circuit, ibinabalik ng Qiskit Runtime ang tumpak na impormasyon ng timing ng circuit sa loob ng mga resulta ng job kapag hiniling.

> **Note:** - Ito ay isang experimental na function. Nasa preview release status ito at samakatuwid ay maaaring magbago.
> - Ang function na ito ay nalalapat lamang sa mga Qiskit Runtime Sampler job.
> - Kahit na ang kabuuang oras ng circuit ay ibinababa sa "compilation" metadata, HINDI ito ang oras na ginagamit para sa billing (quantum time).
### I-enable ang pagkuha ng timing data
Para i-enable ang pagkuha ng timing data, itakda ang experimental na `scheduler_timing` flag sa `True` kapag pinatatakbo ang primitive job.